In [5]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, r2_score

In [6]:
# Load dataset
df = pd.read_csv("calorie_dataset_with_context_aware_meals.csv")

# Drop output and suggestion columns
X = df.drop(columns=["Daily Calorie Need", "Breakfast Suggestion", "Lunch Suggestion", "Dinner Suggestion", "Snack Suggestion"])
y = df["Daily Calorie Need"]

# Define feature types
categorical_cols = ['Gender', 'Activity Level', 'Dietary Preference', 'Budget Preferences']
numerical_cols = ['Age', 'Height', 'Weight', 'Acne', 'Diabetes', 'Heart Disease',
                  'Hypertension', 'Kidney Disease', 'Weight Gain', 'Weight Loss']


In [7]:
# Preprocessing for categorical features
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Column transformer
preprocessor = ColumnTransformer(
    transformers=[('cat', categorical_transformer, categorical_cols)],
    remainder='passthrough'  # Pass through numerical features
)


In [8]:
# Build the pipeline with Gradient Boosting
model = make_pipeline(
    preprocessor,
    GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
model.fit(X_train, y_train)


,steps,"[('columntransformer', ...), ('gradientboostingregressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [9]:
# Predictions and evaluation
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error:", mae)
print("R² Score:", r2)

Mean Absolute Error: 40.70265217453487
R² Score: 0.9900887131592773


In [10]:
import joblib
# Save the trained pipeline model
joblib.dump(model, "calorie_prediction_model.pkl")
print("Model saved successfully.")

Model saved successfully.


Sentence Transformer

In [11]:
# Load dataset
df = pd.read_csv("calorie_dataset_with_context_aware_meals.csv")

# Combine meal suggestions into a single string per row
df['Meal Plan'] = df['Breakfast Suggestion'] + "; " + df['Lunch Suggestion'] + "; " + df['Dinner Suggestion'] + "; " + df['Snack Suggestion']

# Load sentence-transformers model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Embed all meal plans
meal_embeddings = model.encode(df['Meal Plan'].tolist(), show_progress_bar=True)

C:\Users\sheha\PycharmProjects\AI_Nutritionist\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sheha\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not install

In [12]:
def get_meal_plan_for_calorie(calorie_target):
    # Create a calorie context sentence
    calorie_context = f"A healthy meal plan for a person needing {calorie_target} calories"

    # Embed the query
    query_embedding = model.encode([calorie_context])

    # Compute cosine similarity with all meal plans
    similarities = cosine_similarity(query_embedding, meal_embeddings)[0]

    # Find best match
    best_index = np.argmax(similarities)

    return df.iloc[best_index][['Breakfast Suggestion', 'Lunch Suggestion', 'Dinner Suggestion', 'Snack Suggestion']]

In [13]:
# Example usage
suggested_meals = get_meal_plan_for_calorie(2200)
print("Suggested Meal Plan:")
print(suggested_meals)

Suggested Meal Plan:
Breakfast Suggestion                Egg sandwich
Lunch Suggestion         Rajma chawal with salad
Dinner Suggestion             High-protein pasta
Snack Suggestion        Banana and peanut butter
Name: 112, dtype: object


In [14]:
# Save meal embeddings
np.save("meal_embeddings.npy", meal_embeddings)

# Save the meal plan DataFrame
df[['Breakfast Suggestion', 'Lunch Suggestion', 'Dinner Suggestion', 'Snack Suggestion', 'Meal Plan']].to_csv("meal_suggestion_meal_plans.csv", index=False)

# Save the model
model.save("meal_suggestion_sentence_model")